# Table Builder
Runs the table builder independently.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd() if Path.cwd().name == 'certification_framework' else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [2]:
import json
from scripts.computation.table_builder import build_from_file

NB_DATA = ROOT / 'notebooks' / 'data'
phase1_path = NB_DATA / 'parsed_context.json'
result = build_from_file(phase1_path)

# Save to notebook-local data
(NB_DATA / 'tables.json').write_text(
    json.dumps(result, indent=2, default=str, ensure_ascii=False), encoding='utf-8'
)

print(f'=== Tables ({len(result["tables"])}) ===')
for name, table in result['tables'].items():
    headers = table.get('headers', [])
    rows = table.get('rows', [])
    print(f'\n  {name} ({len(rows)} rows, {len(headers)} cols)')
    print(f'    Headers: {headers}')
    for row in rows[:3]:
        print(f'    {row}')

=== Tables (13) ===

  judge_models (4 rows, 4 cols)
    Headers: ['Judge', 'Model', 'Provider', 'Role']
    ['Judge 1', 'gpt-4.1', 'Azure OpenAI', 'Primary evaluator — reasoning, response quality, RAI compliance']
    ['Judge 2', 'gpt-4.1-mini', 'Azure OpenAI', 'Secondary evaluator — security compliance, hallucination detection']
    ['Judge 3', 'gpt-4o', 'Azure OpenAI', 'Tertiary evaluator — cross-validation, safety propagation checks']

  ttd_stats (3 rows, 8 cols)
    Headers: ['Category', 'Runs', 'Mean', 'Median', 'Std Dev', 'P95', 'Min', 'Max']
    ['Application', 5, '366.3s', '455.0s', '205.0s', '584.6s', '60.8s', '584.6s']
    ['Network', 5, '536.6s', '437.1s', '404.5s', '1226.0s', '163.1s', '1226.0s']
    ['Resource', 5, '1364.5s', '1366.7s', '106.0s', '1497.6s', '1241.5s', '1497.6s']

  ttm_stats (3 rows, 8 cols)
    Headers: ['Category', 'Runs', 'Mean', 'Median', 'Std Dev', 'P95', 'Min', 'Max']
    ['Application', 5, '482.8s', '419.3s', '260.3s', '814.7s', '128.8s', '814.7s'

In [3]:
raw = json.loads((ROOT / 'data' / 'input' / 'aggregated_scorecard_output.json').read_text(encoding='utf-8'))

print('=== Validation: Tables vs Input ===')
all_ok = True
cat_count = len(raw['fault_category_scorecards'])

# Tables with one row per category
per_cat_tables = ['ttd_stats', 'ttm_stats', 'detection_rates', 'action_correctness',
                  'reasoning_quality', 'hallucination', 'token_usage']
for tbl_name in per_cat_tables:
    if tbl_name in result['tables']:
        rows = len(result['tables'][tbl_name].get('rows', []))
        ok = 'PASS' if rows == cat_count else 'FAIL'
        if ok == 'FAIL': all_ok = False
        print(f'  {ok} {tbl_name}: {rows} rows (expected {cat_count})')

# All tables must have headers
for tbl_name, table in result['tables'].items():
    has_headers = len(table.get('headers', [])) > 0
    ok = 'PASS' if has_headers else 'FAIL'
    if ok == 'FAIL': all_ok = False
    print(f'  {ok} {tbl_name} has headers: {has_headers}')

print(f'\nResult: {"ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED"}')

=== Validation: Tables vs Input ===
  PASS ttd_stats: 3 rows (expected 3)
  PASS ttm_stats: 3 rows (expected 3)
  PASS detection_rates: 3 rows (expected 3)
  PASS action_correctness: 3 rows (expected 3)
  PASS reasoning_quality: 3 rows (expected 3)
  PASS hallucination: 3 rows (expected 3)
  PASS token_usage: 3 rows (expected 3)
  PASS judge_models has headers: True
  PASS ttd_stats has headers: True
  PASS ttm_stats has headers: True
  PASS detection_rates has headers: True
  PASS safety_summary has headers: True
  PASS action_correctness has headers: True
  PASS reasoning_quality has headers: True
  PASS hallucination has headers: True
  PASS rai_compliance has headers: True
  PASS security_compliance has headers: True
  PASS token_usage has headers: True
  PASS limitations has headers: True
  PASS recommendations has headers: True

Result: ALL CHECKS PASSED
